<h2>Food (v0.03) - 'Food (v0.03).ipynb'</h2>

<u><h5>Context</h5></u>

The notebook aims to isolate NZF impacts modelled for individual commodities with 'CPC' codes forwarded over by David Forengie on 23rd September 2026.

The objectives of the notebook are:

<ol>
    <li>
        To develop the representation of CPC codes provided by David, as its not completely clear how these work (e.g. where some CPC codes are only 3 digits, whereas others are 5). 
    </li>
    <li>
        To read-in 'v0.64' of the 'Impact' analysis results for the country in question, then isolate these records for the CPC codes identified in the step above.
    </li>
</ol>

<u><h5>Version Documentation</h5></u>

'v0.01' outlined an initial method for completing the process. 'v0.02' refined the method and developed a fully functioning notebook, utilising results from the 'Impact' notebooks which now include CPC codes as fields in the output data tables. Upon inspection, it was found that the identification of CPC codes wasn't functioning properly, where CPC codes specified by David wouldn't show in the dataset, yet HSa4 codes given in the CPC manual would show. This likely explains much of the losses observed when pulling by CPC codes, so 'v0.03' utilises a new version of the codes sent through David which have been mapped to the HS system ahead of time. 

In [4]:
# Import packages
import os
import base64
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 250)
pd.set_option('display.max_rows', 250)
pd.options.mode.chained_assignment = None
from IPython.display import HTML
import matplotlib.pyplot as plt
import matplotlib.style
plt.style.use('seaborn-v0_8')

def download_link(df, title="Download CSV file", filename="data.csv"):
    csv = df.to_csv()
    b64 = base64.b64encode(csv.encode())
    payload = b64.decode()
    html = '<a download="{filename}" href="data:text/csv;base64,{payload}" target="_blank">{title}</a>'
    html = html.format(payload=payload,title=title,filename=filename)
    return HTML(html)

# define function for counting values in each data field
trade_im_field_count = lambda field: trade_im[field].value_counts().to_frame().reset_index()
trade_ex_field_count = lambda field: trade_ex[field].value_counts().to_frame().reset_index()

version = "v0.03"
input_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/"
output_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/food/"

# 1. Set up the notebook

## 1.1 Let's set the country we're interested in and set up a results folder.

In [2]:
# Select one of our countries to work on...
iso_code = "585"
iso_country = "Palau"
year = 2017
import_portfolio = "h4_ag6_by_sea_M.csv" # Imports, '_M' | Foreign Imports, '_FM' | Re-Imports, '_RM'
export_portfolio = "h4_ag6_by_sea_X.csv" # Exports, '_X' | Domestic Exports, '_DX' | Re-exports, '_RX'
gdp = 290*1e6

# Create output Journey directory folder if necessary
os.makedirs(output_dir+"{0}/".format(iso_country)) if not os.path.exists(output_dir+"{0}/".format(iso_country)) else "Output directory already exists."

## 1.2 Import and Format 'Impact Analysis' Results

In [3]:
impact_im_c = [
    "trade_idx", "reporterCode", "reporterDesc", "partnerCode", "partnerDesc", "flowDesc", "motDesc", 
    "HS6", "HS6_desc", "HS4", "HS4_desc", "HS2", "HS2_desc", "cpc", "cpc_desc", "eora_desc", "gtap", "gtap_desc", 
    "cifvalue", "fobvalue", "primaryValue", "qtyUnitAbbr", "qty", "netWgt", "vol_kg_vtw", 
    "sea_dist_km", "sea_dist_nm", "tnm", "fac_vessel_types", "fac_vessel_types_str", 
    #"TandT_match_desc", "TandT_ad_val_pct", "TandT_N", "TandT_ciffob", "TandT_cifvalue", "TandT_fobvalue", "cost_int_match_desc",
    "bau_usd_26", "levy_usd_26", "asis_usd_26", "brazil_usd_26", "liscr_usd_26", "direct_usd_26", 
    "bau_usd_29", "levy_usd_29", "asis_usd_29", "brazil_usd_29", "liscr_usd_29", "direct_usd_29", 
    "bau_usd_32", "levy_usd_32", "asis_usd_32", "brazil_usd_32", "liscr_usd_32", "direct_usd_32", 
    "bau_usd_35", "levy_usd_35", "asis_usd_35", "brazil_usd_35", "liscr_usd_35", "direct_usd_35", 
    "bau_usd_38", "levy_usd_38", "asis_usd_38", "brazil_usd_38", "liscr_usd_38", "direct_usd_38", 
    "bau_usd_41", "levy_usd_41", "asis_usd_41", "brazil_usd_41", "liscr_usd_41", "direct_usd_41", 
    "bau_usd_44", "levy_usd_44", "asis_usd_44", "brazil_usd_44", "liscr_usd_44", "direct_usd_44", 
    "bau_usd_47", "levy_usd_47", "asis_usd_47", "brazil_usd_47", "liscr_usd_47", "direct_usd_47", 
    "bau_usd_50", "levy_usd_50", "asis_usd_50", "brazil_usd_50", "liscr_usd_50", "direct_usd_50"	
]

impact_im_d = {
    "reporterCode":"str", "partnerCode":"str", 
    "HS6":"str", "HS4":"str", "HS2":"str", "HS2":"str", "gtap":"str", "cpc":"str"
}

impact_im = pd.read_csv(
    input_dir + "impact/{0}/impact_im_v0.64.csv".format(iso_country), 
    dtype=impact_im_d,
    usecols=impact_im_c
)[impact_im_c]

print("Aggregate trade portfolio statistics for '{0}': \n".format(iso_country))
print("Number of trade records present in dataset: {0}".format(impact_im.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(impact_im.partnerCode.unique())))
print("Modes of transport present in dataset: {0}".format(impact_im.motDesc.unique()))
print("Total value within the dataset: US${0}m".format(np.round(impact_im.cifvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(impact_im.vol_kg_vtw.sum() / 1e6, 1)))

impact_im["cpc_cl"] = [str(i).split('.')[0] for i in impact_im.cpc.values]
impact_im = impact_im.drop(columns="trade_idx").reset_index(drop=True).reset_index().rename(columns={"index":"trade_idx"})
display(impact_im.head(2))

Aggregate trade portfolio statistics for 'Palau': 

Number of trade records present in dataset: 7671
Number of partner countries present in dataset: 61
Modes of transport present in dataset: ['Sea']
Total value within the dataset: US$130.6m
Total weight within the dataset: 117.6 million kilograms


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl
0,0,585,Palau,020,Andorra,Import,Sea,40120,040120 | Dairy produce; milk,0401,0401 | Milk and cream; not,04,04 | Dairy produce; birds,2211,Processed liquid milk,Agriculture,22,MIL - Dairy products,12.25,12.25,12.25,kg,35.0,35.0,15.28065,14578.0,9058.346438,138.417421,['Container'],['Container'],0.723484,0.722221,0.722344,0.722344,0.722911,0.722911,0.756801,1.341289,0.834776,0.775012,0.777117,0.796580,0.741041,1.313587,0.878654,0.804834,0.786365,0.821980,0.727219,1.428437,1.089574,0.952903,0.881615,0.856746,0.713653,1.518779,1.249781,1.171151,0.943883,0.918217,0.700343,1.53907,1.470445,1.483088,1.023267,0.964558,0.686766,1.507436,1.444585,1.582755,1.170753,0.982237,0.673976,1.344669,1.401503,1.468629,1.100931,0.993842,0.660934,1.235194,1.255607,1.566207,1.293907,1.000424,2211
1,1,585,Palau,020,Andorra,Import,Sea,340319,340319 | Lubricating preparat,3403,3403 | Lubricating preparat,34,"34 | Soap, organic surfac",3543,Lubricating preparations and preparations of a...,"Petroleum, Chemical and Non-Metallic Mineral P...",33,"CRP - Chemical, rubber, plastic products",9.00,9.00,9.00,kg,3.0,3.0,2.15280,14578.0,9058.346438,19.500808,['Container'],['Container'],0.101927,0.101749,0.101767,0.101767,0.101847,0.101847,0.106621,0.188966,0.117607,0.109187,0.109483,0.112225,0.104401,0.185064,0.123788,0.113388,0.110786,0.115804,0.102454,0.201244,0.153504,0.134249,0.124206,0.120702,0.100542,0.213972,0.176074,0.164996,0.132978,0.129362,0.098667,0.21683,0.207162,0.208943,0.144162,0.135891,0.096754,0.212374,0.203519,0.222985,0.164940,0.138381,0.094952,0.189442,0.197449,0.206906,0.155104,0.140017,0.093115,0.174019,0.176895,0.220654,0.182291,0.140944,3543


# 2. Subset the 'Impact Analysis' results for food security-related CPC codes

## 2.1 Let's get a sense of how much of import value is associated with food-relevant goods across classification systems

### 2.1.1 Harmonised System (HS01-23)

In [5]:
#print("HS groupings: \n\n{0}\n".format(pd.Series(impact_im.HS2_desc.unique()).sort_values().values))
HSa2_good_groups = [
    "01", "02", "03", "04", "05", "07", "08", # "06", 
    "09", "10", "11", "12", "13", "14", "15", "16", 
    "17", "18", "19", "20", "21", "22", "23"
]

impact_im_HSa2_food = impact_im[(impact_im.HS2.isin(HSa2_good_groups))]
impact_im_HSa2_food_g = impact_im_HSa2_food.groupby(["HS2_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_HSa2_food_g["pct_val"] = round(100 * impact_im_HSa2_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_HSa2_food_g["pct_vol"] = round(100 * impact_im_HSa2_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_HSa2_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_HSa2_food_g.round(1)
))

Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): $32.8m (25.1%), ... 

                            cifvalue  vol_kg_vtw  pct_val  pct_vol
HS2_desc                                                          
22 | Beverages, spirits a  6467082.2  11977485.9      5.0     10.2
02 | Meat and edible meat  4856869.4   1623218.7      3.7      1.4
19 | Preparations of cere  3166741.5   1293344.5      2.4      1.1
16 | Meat, fish, crustace  2450097.6    528107.7      1.9      0.4
04 | Dairy produce; birds  2406806.4   1520544.5      1.8      1.3
21 | Miscellaneous edible  2253873.9    827672.8      1.7      0.7
20 | Preparations of vege  1907828.6   1282005.3      1.5      1.1
10 | Cereals               1691768.0   2989406.3      1.3      2.5
07 | Vegetables and certa  1472336.4   1869051.3      1.1      1.6
09 | Coffee, tea, mate an  1078602.0    316577.4      0.8      0.3
08 | Fruit and nuts, edib   924480.2    831184.1      0.7      0.7
18 | Cocoa and cocoa prep   9

### 2.1.2 EORA groupings ('Food & Beverages' and 'Fishing')

In [6]:
print("EORA groupings: \n\n{0}\n".format(pd.Series(impact_im.eora_desc.unique()).sort_values().values))
eora_good_groups = ["Food & Beverages", "Agriculture", "Fishing"]

impact_im_eora_food = impact_im[(impact_im.eora_desc.isin(eora_good_groups))]
impact_im_eora_food_g = impact_im_eora_food.groupby(["eora_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_eora_food_g["pct_val"] = round(100 * impact_im_eora_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_eora_food_g["pct_vol"] = round(100 * impact_im_eora_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_eora_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_eora_food_g.round(1)
))

EORA groupings: 

['Agriculture' 'Electrical and Machinery' 'Fishing' 'Food & Beverages'
 'Metal Products' 'Mining and Quarrying' 'Other Manufacturing'
 'Petroleum, Chemical and Non-Metallic Mineral Products'
 'Textiles and Wearing Apparel' 'Transport Equipment' 'Wood and Paper' nan]

Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): $32.1m (24.6%), ... 

                    cifvalue  vol_kg_vtw  pct_val  pct_vol
eora_desc                                                 
Food & Beverages  18574806.1  16865702.4     14.2     14.3
Agriculture       12894028.8  10066928.5      9.9      8.6
Fishing             621284.6    180874.8      0.5      0.2



### 2.1.3 GTAP (16x headings)

In [7]:
#print("GTAP groupings: \n\n{0}\n".format(pd.Series(impact_im.gtap_desc.unique()).sort_values().values))
gtap_food_groups = [
    "GRO - Cereal grains n.e.c.", "PDR - Paddy rice", "PCR - Processed rice", "SGR - Sugar", 
    "CMT - Bovine meat prods", "CTL - Bovine cattle, sheep and goats, horses", "FSH - Fishing", "OMT - Meat products n.e.c.",
    "MIL - Dairy products", "OFD - Food products n.e.c.", "V_F - Vegetables, fruit, nuts", "OSD - Oil seeds", 
    "OCR - Crops n.e.c.", "PFB - Plant-based fibers", "VOL - Vegetable oils and fats", 
    "OAP - Animal products n.e.c."
]

impact_im_gtap_food = impact_im[(impact_im.gtap_desc.isin(gtap_food_groups))]
impact_im_gtap_food_g = impact_im_gtap_food.groupby(["gtap_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_gtap_food_g["pct_val"] = round(100 * impact_im_gtap_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_gtap_food_g["pct_vol"] = round(100 * impact_im_gtap_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via GTAP: ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_gtap_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_gtap_food_g.head(15).round(1)
))

Value of food imports identified via GTAP: $24.9m (19.0%), ... 

                                 cifvalue  vol_kg_vtw  pct_val  pct_vol
gtap_desc                                                              
OFD - Food products n.e.c.     11034995.4   5558458.3      8.5      4.7
OMT - Meat products n.e.c.      3789325.0   1395468.7      2.9      1.2
CMT - Bovine meat prods         2581596.7    560600.8      2.0      0.5
MIL - Dairy products            2398397.4   1405704.9      1.8      1.2
V_F - Vegetables, fruit, nuts   2088990.2   2472860.7      1.6      2.1
PCR - Processed rice            1654660.6   2874145.4      1.3      2.4
VOL - Vegetable oils and fats    627806.8    392666.8      0.5      0.3
OCR - Crops n.e.c.               497766.1    164729.4      0.4      0.1
SGR - Sugar                       85294.2    187763.6      0.1      0.2
FSH - Fishing                     49263.3     11399.2      0.0      0.0
OAP - Animal products n.e.c.      27684.5      8995.2      0.0      0.0

## 2.2 Comparison of results

In [8]:
print("Share of total import value associated with food-relevant goods by classification system...\n\nHSa2 {0}% \nGTAP {1}% \nEORA {2}%".format(
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))

Share of total import value associated with food-relevant goods by classification system...

HSa2 25.1% 
GTAP 19.0% 
EORA 24.6%


# 3. Take HSa4 codes associated with specific CPCs

## 3.1 Set HSa4 Codes we're interested in

In [9]:
hsa4_codes_food = {
    "Dairy": [
        "0401", "0402", "0403", "0404", "0405", "0406",
    ],
    "Fats and Oils": [
        "1501", "1502", "1503", "1504", "1505", "1506", "1507", "1508", "1509", "1510"
        "1511", "1512", "1513", "1514", "1515", "1516", "1517"
    ],
    "Fruits": [
        "0709", "0801", "0802", "0803", "0804", "0805", "0806", "0807", "0808", "0809", 
        "0810", "0813", "2009"
    ],
    "Meat and Prep": [
        "0201", "0203", "0204", "0205", "0207", "0208", "0209", "2301"
    ],
    "Processed Cereals": [
        "1006", "1101", "1102", "1104", "1105", "1106", "1107", "1108", "1109", "1302",
        "1602", "1901", "1902", "1903", "1904", "1905", "2005", "2007", "2104", "2106", 
        "3505"
    ],
    "Sugar and Honey": [
        "1701", "1702", "1703", "1704", "2006"
    ],
    "Unprocessed Cereals": [
        "1001", "1002", "1003", "1004", "1005", "1007", "1008"
    ],
    "Vegetables": [
        "0701", "0702", "0703", "0704", "0705", "0706", "0707", "0708", "0709", "0710",
        "0712", "0713", "0714", "1209", "2009"
    ]
}

## 3.2 Identify 'Food Security' groups of interest via CPC

In [10]:
impact_im["food_group"] = np.nan

for food_group in list(hsa4_codes_food.keys()):
    impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group

display(impact_im.head(2))

/var/folders/_k/81z8ryjd28lc4f0bb4dm7gkw0000gn/T/ipykernel_27840/1412186970.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Dairy' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl,food_group
0,0,585,Palau,020,Andorra,Import,Sea,40120,040120 | Dairy produce; milk,0401,0401 | Milk and cream; not,04,04 | Dairy produce; birds,2211,Processed liquid milk,Agriculture,22,MIL - Dairy products,12.25,12.25,12.25,kg,35.0,35.0,15.28065,14578.0,9058.346438,138.417421,['Container'],['Container'],0.723484,0.722221,0.722344,0.722344,0.722911,0.722911,0.756801,1.341289,0.834776,0.775012,0.777117,0.796580,0.741041,1.313587,0.878654,0.804834,0.786365,0.821980,0.727219,1.428437,1.089574,0.952903,0.881615,0.856746,0.713653,1.518779,1.249781,1.171151,0.943883,0.918217,0.700343,1.53907,1.470445,1.483088,1.023267,0.964558,0.686766,1.507436,1.444585,1.582755,1.170753,0.982237,0.673976,1.344669,1.401503,1.468629,1.100931,0.993842,0.660934,1.235194,1.255607,1.566207,1.293907,1.000424,2211,Dairy
1,1,585,Palau,020,Andorra,Import,Sea,340319,340319 | Lubricating preparat,3403,3403 | Lubricating preparat,34,"34 | Soap, organic surfac",3543,Lubricating preparations and preparations of a...,"Petroleum, Chemical and Non-Metallic Mineral P...",33,"CRP - Chemical, rubber, plastic products",9.00,9.00,9.00,kg,3.0,3.0,2.15280,14578.0,9058.346438,19.500808,['Container'],['Container'],0.101927,0.101749,0.101767,0.101767,0.101847,0.101847,0.106621,0.188966,0.117607,0.109187,0.109483,0.112225,0.104401,0.185064,0.123788,0.113388,0.110786,0.115804,0.102454,0.201244,0.153504,0.134249,0.124206,0.120702,0.100542,0.213972,0.176074,0.164996,0.132978,0.129362,0.098667,0.21683,0.207162,0.208943,0.144162,0.135891,0.096754,0.212374,0.203519,0.222985,0.164940,0.138381,0.094952,0.189442,0.197449,0.206906,0.155104,0.140017,0.093115,0.174019,0.176895,0.220654,0.182291,0.140944,3543,NaN


## 3.3 Inspect the output

In [11]:
impact_im_cpc_food = impact_im[(~impact_im.food_group.isna())]
impact_im_cpc_food_g = impact_im_cpc_food.groupby("food_group").agg({"HS6_desc":"count", "cifvalue":"sum", "vol_kg_vtw":"sum"}).reset_index().sort_values(by="cifvalue", ascending=False).round(2).rename(columns={"HS6_desc":"N_com"}).reset_index(drop=True)
impact_im_cpc_food_g["pct_val"] = round(100 * impact_im_cpc_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_cpc_food_g["pct_vol"] = round(100 * impact_im_cpc_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Proportion of imports identified as food security-relevant:\n")
print("Value: ${0}m of ${1}m ({2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / 1e6, 1),
    round(impact_im.cifvalue.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))
print("Volume: {0} of {1} million kilograms ({2}%)".format(
    round(impact_im_cpc_food.vol_kg_vtw.sum() / 1e6, 1),
    round(impact_im.vol_kg_vtw.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.vol_kg_vtw.sum() / impact_im.vol_kg_vtw.sum(), 1)
))
print("Value-to-Weight: {0} from {1} US$ per kg (+{2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum(), 2),
    round(impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum(), 2),
    round(100 * ((impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum()) - (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum())) / (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum()), 1)
))
print("Average distance to market (volume-weighted): {0} from {1} nm (+{2}%)".format(
    round((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000), 1),
    round((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000), 1),
    round(100 * (((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000)) - ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000))) / \
          ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000)), 1)
))
print("\n{0}\n".format(impact_im_cpc_food_g.head(15).round(1)))

impact_im_cpc_food_g.to_csv(output_dir + "{0}/food_total_usd_{1}.csv".format(iso_country, version), index=False)

Proportion of imports identified as food security-relevant:

Value: $15.6m of $130.6m (12.0%)
Volume: 12.7 of 117.6 million kilograms (10.8%)
Value-to-Weight: 1.23 from 1.11 US$ per kg (+11.2%)
Average distance to market (volume-weighted): 5328.8 from 2969.1 nm (+79.5%)

            food_group  N_com   cifvalue  vol_kg_vtw  pct_val  pct_vol
0    Processed Cereals    256  7232799.0   5672263.5      5.5      4.8
1           Vegetables    153  2377803.8   2623445.2      1.8      2.2
2        Meat and Prep     19  2108624.2   1118594.5      1.6      1.0
3                Dairy     63  1759297.1   1164139.3      1.3      1.0
4               Fruits     82   884569.9    812889.5      0.7      0.7
5      Sugar and Honey     57   630116.5    794351.2      0.5      0.7
6        Fats and Oils     54   623365.1    386235.0      0.5      0.3
7  Unprocessed Cereals     14    28279.5     99926.7      0.0      0.1



## 3.3 Derive results for 2029, 2041 and 2050

### 3.3.1 Total FEM costs by scenario (US$)

In [12]:
print("\nTotal FEM costs by scenario (US$)...\n")

d = {"food_group": list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["bau", "levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            round(100 * impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum(), 1) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_usd_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$)...



,food_group,bau_29,levy_29,asis_29,brazil_29,liscr_29,direct_29,bau_41,levy_41,asis_41,brazil_41,liscr_41,direct_41,bau_50,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,3731568.3,6601214.5,4113555.8,3820338.1,3830564.9,3926742.0,3454359.3,7574633.9,7237919.6,7301983.4,5040471.7,4751288.7,3260858.8,6083477.7,6183777.9,7704998.6,6366794.0,4927314.5
1,Fats and Oils,1139084.6,1978457.5,1246954.4,1160513.3,1163631.8,1195334.8,1057827.1,2260799.0,2164644.0,2194766.1,1521438.3,1435702.5,1001107.3,1829104.0,1862303.1,2292614.6,1895440.4,1486900.7
2,Fruits,2817188.9,4981832.8,3105210.5,2884059.1,2891756.6,2964396.1,2608080.2,5716478.1,5462522.0,5511132.5,3804642.4,3586345.3,2462116.9,4591777.6,4667435.8,5814389.7,4804770.4,3719111.7
3,Meat and Prep,3665114.5,6495732.9,4042738.8,3753306.8,3763499.9,3857760.0,3391692.5,7453564.1,7121220.9,7182449.8,4955581.6,4671259.0,3200836.9,5981922.1,6080780.3,7584988.6,6266264.9,4844953.4
4,Processed Cereals,14641548.1,25822082.9,16119979.8,14978474.9,15018545.7,15400290.0,13560945.5,29640334.7,28318441.5,28589117.3,19750136.4,18610168.4,12806657.2,23832332.9,24242012.0,30119154.7,24896145.3,19294001.0
5,Sugar and Honey,1043320.8,1803197.3,1138428.0,1061157.3,1064150.7,1093939.4,969544.8,2075858.4,1979622.3,2009450.6,1395259.0,1310592.6,918047.3,1681532.1,1721286.6,2093212.6,1733668.7,1355906.5
6,Unprocessed Cereals,222576.3,383246.2,242394.0,226074.6,226731.7,233228.2,206956.0,441738.4,420887.1,427765.1,297300.5,278949.1,196052.6,358291.0,367505.6,444555.0,368347.5,288454.0
7,Vegetables,8381549.4,14717789.5,9215727.0,8570668.6,8592486.4,8810822.5,7769532.3,16894901.4,16149866.9,16311527.3,11276434.1,10629835.0,7341715.6,13602444.7,13826206.8,17143545.6,14177011.6,11017695.6


### 3.3.2 Total FEM costs by scenario (% cif)

In [13]:
print("\nTotal FEM costs by scenario (% cif)...\n")

d = {"food_group":list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            np.round(100 * (impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum() - impact_im[(impact_im.food_group == hsa4_codes_food_key)]["bau_usd_{1}".format(scenario, year)].sum()) / \
                     impact_im[(impact_im.food_group == hsa4_codes_food_key)].cifvalue.sum(), 2) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_pct_cif_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (% cif)...



,food_group,levy_29,asis_29,brazil_29,liscr_29,direct_29,levy_41,asis_41,brazil_41,liscr_41,direct_41,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,1.63,0.22,0.05,0.06,0.11,2.34,2.15,2.19,0.90,0.74,1.60,1.66,2.53,1.77,0.95
1,Fats and Oils,1.35,0.17,0.03,0.04,0.09,1.93,1.78,1.82,0.74,0.61,1.33,1.38,2.07,1.43,0.78
2,Fruits,2.45,0.33,0.08,0.08,0.17,3.51,3.23,3.28,1.35,1.11,2.41,2.49,3.79,2.65,1.42
3,Meat and Prep,1.34,0.18,0.04,0.05,0.09,1.93,1.77,1.80,0.74,0.61,1.32,1.37,2.08,1.45,0.78
4,Processed Cereals,1.55,0.20,0.05,0.05,0.10,2.22,2.04,2.08,0.86,0.70,1.52,1.58,2.39,1.67,0.90
5,Sugar and Honey,1.21,0.15,0.03,0.03,0.08,1.76,1.60,1.65,0.68,0.54,1.21,1.27,1.86,1.29,0.69
6,Unprocessed Cereals,5.68,0.70,0.12,0.15,0.38,8.30,7.56,7.81,3.19,2.55,5.74,6.06,8.79,6.09,3.27
7,Vegetables,2.66,0.35,0.08,0.09,0.18,3.84,3.52,3.59,1.47,1.20,2.63,2.73,4.12,2.87,1.55
